# Project: Forecasting sales and catching anomalies

*ML Academy — complete solution*

**Goal.** Forecast the last 8 weeks better than the seasonal naive baseline, check the result with a backtest, and use the forecast errors to find the days that went wrong.

**The data.** Two years of simulated daily sales for one shop, with growth, a weekly pattern, planned promotions and 12 hidden problem days (till outages and data errors). The data is generated in the notebook, so the true anomalies are known and your detector can be scored honestly.

**Uses:** Lesson 22, Lesson 23, Lesson 25, Lesson 26.

This notebook contains the full solution with outputs. Try the starter notebook first!

Requirements: `pip install numpy pandas scikit-learn matplotlib scipy` — the data is simulated in the first step, so no download is needed.

In [1]:
# Run this cell first: it defines check(), which tells you whether each step is right.
import numpy as np
def check(get, want, hint=''):
    try:
        got = get()
        if isinstance(want, str):
            ok = str(got) == want
        else:
            ok = np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=1e-6)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


## Step 1. Load the sales history

The data is simulated (so we know the truth): two years of daily sales for one shop, with growth, a weekly pattern, promotions, and 12 days where something went wrong (a till outage or a data error). The list `true_anomalies` is hidden from your models; it's only used at the end to score your detector.

In [2]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(7)
days = pd.date_range("2024-01-01", periods=730, freq="D")
weekly = np.array([0.85, 0.9, 0.95, 1.0, 1.1, 1.35, 1.2])[days.dayofweek]
promo = (rng.random(730) < 0.08).astype(int)
sales = (200 + 0.12 * np.arange(730)) * weekly * (1 + 0.25 * promo) * rng.lognormal(0, 0.06, 730)
true_anomalies = np.sort(rng.choice(np.arange(60, 730), 12, replace=False))
sales[true_anomalies] *= rng.choice([0.35, 1.9], 12)
df = pd.DataFrame({"sales": sales.round(1), "promo": promo}, index=days)
print(df.shape)
df.head()

(730, 2)


,sales,promo
2024-01-01,166.6,0
2024-01-02,162.7,0
2024-01-03,194.5,0
2024-01-04,191.6,0
2024-01-05,216.2,0


In [3]:
check(lambda: df.shape, [730, 2], hint='Run the cell as given.')

✅ Correct!


## Step 2. Split by time and set baselines

Keep the last 8 weeks (56 days) as the test period. Compute the MAE of the **seasonal naive** forecast (same day last week) on the test period. Any model must beat it.

In [4]:
test_start = df.index[-56]
test = df.index >= test_start
seasonal_naive = df.sales.shift(7)
naive_mae = float((df.sales[test] - seasonal_naive[test]).abs().mean())
print("seasonal naive MAE:", round(naive_mae, 1))

seasonal naive MAE: 40.7


In [5]:
check(lambda: round(naive_mae, 1), 40.7, hint='Use df.sales.shift(7).')

✅ Correct!


## Step 3. Build past-only features

Lags of 7, 14 and 28 days, a 7-day rolling mean of the previous days (shift *before* rolling), day-of-week dummies and the promotion flag, which is planned in advance so it's known.

In [6]:
X = pd.DataFrame({f"lag_{l}": df.sales.shift(l) for l in [7, 14, 28]}, index=df.index)
X["roll_7"] = df.sales.shift(1).rolling(7).mean()
X["promo"] = df.promo
X = X.join(pd.get_dummies(pd.Series(df.index.dayofweek, index=df.index), prefix="dow", dtype=float)).dropna()
y = df.sales[X.index]
print(X.shape)

(702, 12)


In [7]:
check(lambda: X.shape, [702, 12], hint='Shift by 1 before .rolling(7).mean().')

✅ Correct!


## Step 4. Fit a model and compute MASE

Train Ridge on everything before the test period, forecast the test period and compute the MAE. Then the MASE: divide by the seasonal naive MAE *on the training data*.

In [8]:
from sklearn.linear_model import Ridge
tr, te = X.index < test_start, X.index >= test_start
model = Ridge(1.0).fit(X[tr], y[tr])
pred = model.predict(X[te])
model_mae = float(np.abs(y[te] - pred).mean())
train_naive_mae = float((df.sales - df.sales.shift(7))[df.index < test_start].abs().mean())
mase = model_mae / train_naive_mae
print(f"model MAE {model_mae:.1f}  ·  MASE {mase:.2f}")

model MAE 29.0  ·  MASE 0.98


In [9]:
check(lambda: round(model_mae, 1), 29.0, hint='Ridge(1.0).fit(X[tr], y[tr]).')
check(lambda: round(mase, 2), 0.98, hint='MAE divided by the training seasonal naive MAE.')

✅ Correct!
✅ Correct!


## Step 5. Backtest

One test window can be lucky. Use `TimeSeriesSplit(n_splits=5, test_size=56)` on the rows before the test period and average the MAE.

In [10]:
from sklearn.model_selection import TimeSeriesSplit
maes = []
for a, b in TimeSeriesSplit(n_splits=5, test_size=56).split(X[tr]):
    m = Ridge(1.0).fit(X[tr].iloc[a], y[tr].iloc[a])
    maes.append(float(np.abs(y[tr].iloc[b] - m.predict(X[tr].iloc[b])).mean()))
backtest_mae = float(np.mean(maes))
print("backtest MAEs:", [round(v, 1) for v in maes], " average", round(backtest_mae, 1))

backtest MAEs: [19.5, 19.8, 22.7, 22.0, 23.4]  average 21.5


In [11]:
check(lambda: round(backtest_mae, 1), 21.5, hint='np.mean(maes).')

✅ Correct!


## Step 6. Flag anomalies from forecast errors

A day is suspicious when it's far from what the model expected. Fit on all the data, compute each day's error, scale it robustly (median and MAD, which outliers can't inflate) and flag days with a robust z-score above 4. Then compare with the hidden truth.

In [12]:
full = Ridge(1.0).fit(X, y)
resid = y - full.predict(X)
mad = float(np.median(np.abs(resid - resid.median())))
z = (resid - resid.median()) / (1.4826 * mad)
flagged = np.where(np.abs(z) > 4)[0] + (len(df) - len(X))
hits = len(set(flagged) & set(true_anomalies))
precision, recall = hits / max(len(flagged), 1), hits / len(true_anomalies)
print(f"flagged {len(flagged)} days · precision {precision:.2f} · recall {recall:.2f}")

flagged 12 days · precision 1.00 · recall 1.00


In [13]:
check(lambda: len(flagged), 12, hint='Use (resid − median) / (1.4826 × MAD).')
check(lambda: round(recall, 2), 1.0, hint='Hits divided by the 12 true anomalies.')

✅ Correct!
✅ Correct!


## Go further

- Make the anomalies smaller (multiply by 0.7 and 1.4 instead of 0.35 and 1.9). How do precision and recall change, and which threshold works best?
- Replace Ridge with gradient boosting on the same features (Lesson 13). Does it beat Ridge on the growing series? Try predicting the change from last week instead.
- Forecast 14 days ahead recursively (Lesson 26) and report the MAE by horizon.